In [1]:
import copy
import json
from pathlib import Path
import random

import pandas as pd
import torch
import plotly.express as px

from pivotal_tokens.hf.loading import load_model, load_tokenizer
from pivotal_tokens.constants import get_data_dir, get_artifacts_dir
from pivotal_tokens.extractor import THINKING_START_TOKEN, THINKING_END_TOKEN, TRUNCATED_TAG


EXP41_DIR = get_data_dir() / "experiments" / "exp4.1-likelihood-tokens" / "exp4.1.1-qwen3-1.7b-likelihood-tokens"
EXP41_RESULTS_FILE = EXP41_DIR / "data" / "loglikelihood_pivotal_tokens.csv"

EXP22_DIR = get_data_dir() / "experiments" / "exp2.2-alt-tokens-on-pivotal-positions" / "exp2.2.1-qwen3-1.7b-alt-tokens-on-pivotal-positions"
EXP22_RESULTS_FILE = EXP22_DIR / "data" / "alternative_tokens.csv"

TRACES_FILE = get_artifacts_dir() / 'exp1.1.1-qwen3-1.7b-traces.csv'

PREPROCESSED_FILE = Path("data/preprocessed.csv")


QWEN3_1_7B_MODEL_ID = 'Qwen/Qwen3-1.7B'
DEVICE = "cuda"


SYSTEM_PROMPT = (
    'Answer the following question directly, strictly without chain-of-thought after "</think>". '
    'The reasoning trace may be truncated; if so, it will end with "<truncated>" placed right before '
    'the closing "</think>". In that case, use the partial reasoning to make your best guess at the '
    'answer. Keep the answer short (e.g., "yes" or "no" for binary questions, a person\'s full name '
    'if the question expects a person, a country name if it asks about a country, etc.). Output the '
    'answer strictly in the format "Answer: `<answer>`" with no extra text.'
)


torch.set_grad_enabled(False)

In [2]:
def parse_token_ids(value: object) -> list[int]:
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return []
    return []

prep_df = pd.read_csv(PREPROCESSED_FILE)
prep_df = prep_df.drop_duplicates()
prep_df["token_ids"] = prep_df["token_ids"].apply(parse_token_ids)

In [3]:
if True:
    model = load_model(model_id=QWEN3_1_7B_MODEL_ID, device=DEVICE, use_flash_attention=True)
    tokenizer = load_tokenizer(model_id=QWEN3_1_7B_MODEL_ID)
    model.eval()

    print(f"Model loaded: {QWEN3_1_7B_MODEL_ID}")
    print(f"Device: {next(model.parameters()).device}")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen3-1.7B
Device: cuda:0


In [4]:
def sample_random_tokens(tokenizer, n: int, skip_special: bool = True) -> str:
    vocab_size = tokenizer.vocab_size
    special_ids = set(tokenizer.all_special_ids) if skip_special else set()
    
    eligible = [i for i in range(vocab_size) if i not in special_ids]
    token_ids = random.choices(eligible, k=n)
    
    return tokenizer.decode(token_ids)

In [15]:
selected_sample_id = "5a7b7ff75542995eb53be93d"
sample_row = prep_df[prep_df["sample_id"] == selected_sample_id].iloc[0]
print(f"Query: {sample_row['query']}")
print(f"Ground truth: {sample_row['ground_truth']}")

Query: Who was born first, Arthur Conan Doyle or Penelope Lively?
Ground truth: Arthur Ignatius Conan Doyle


In [16]:
traces_df = pd.read_csv(TRACES_FILE)
trace_row = traces_df[traces_df["id"] == selected_sample_id].iloc[0]
reasoning_trace = trace_row["trace"]

query = sample_row["query"]

expected_answer = sample_row['ground_truth']
# expected_answer = "Arthur Conan Doyle"
paraphrased_answer = "Arthur Conan Doyle"
wrong_answer = "Penelope Lively"
# random_answer = sample_random_tokens(tokenizer, n=5)

print(f"Ground truth: {expected_answer}")
# print(f"Paraphrased answer: {paraphrased_answer}")
# print(f"Wrong answer: {wrong_answer}")
# print(f"Random answer: {random_answer}")


Ground truth: Arthur Ignatius Conan Doyle


In [18]:
def calc_nnll_scores(query, reasoning_trace, answer):
    if not reasoning_trace.startswith(THINKING_START_TOKEN):
        reasoning_trace = f"{THINKING_START_TOKEN}\n{reasoning_trace}"
    if reasoning_trace.endswith(THINKING_END_TOKEN):
        reasoning_trace = reasoning_trace[: -len(THINKING_END_TOKEN)].rstrip()
        
    prompts = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": query}]
    context = tokenizer.apply_chat_template(
        prompts, tokenize=False, add_generation_prompt=True, enable_thinking=True
    )
    answer_suffix = f"{TRUNCATED_TAG}\n{THINKING_END_TOKEN}\nAnswer: `{answer}`"

    completion_tok = tokenizer(reasoning_trace, return_tensors="pt").to(model.device)
    completion_len = completion_tok.input_ids.shape[1]

    full_text = context + reasoning_trace
    full_tok = tokenizer(full_text, return_tensors="pt", return_attention_mask=False).to(model.device)

    # Indices start from the <thinking> token in order to get nnll of the "no thinking" scenario.
    # Specifically, the case "... <thinking> </thinking> ..."
    completion_start_idx = full_tok.input_ids.shape[1] - completion_len + 1
    completion_end_idx = full_tok.input_ids.shape[1] + 1
    indices = list(range(completion_start_idx, completion_end_idx))

    n_ground_truth_tokens = len(tokenizer.encode(answer))

    past_key_values = None
    prev_t = 0
    last_prefix_logits = None

    tokens = []

    for t in indices:
        new_tokens = full_tok.input_ids[:, prev_t:t]

        if past_key_values is None:
            out_prefix = model(input_ids=new_tokens,
                                use_cache=True)
        else:
            past_length = past_key_values[0][0].shape[2]
            seq_len = new_tokens.shape[1]
            total_len = past_length + seq_len

            position_ids = torch.arange(past_length, total_len, device=model.device).unsqueeze(0)
            attention_mask = torch.ones(1, total_len, device=model.device, dtype=torch.long)

            out_prefix = model(input_ids=new_tokens,
                                past_key_values=past_key_values,
                                position_ids=position_ids,
                                attention_mask=attention_mask,
                                use_cache=True)

        past_key_values = out_prefix.past_key_values
        prev_t = t

        prefix_ids = full_tok.input_ids[:, :t]
        prefix_str = tokenizer.decode(prefix_ids[0], skip_special_tokens=False)

        conditional_context = prefix_str + answer_suffix
        conditional_context_tok = tokenizer(conditional_context, return_tensors="pt").to(model.device)

        suffix_ids = conditional_context_tok.input_ids[:, t:]
        suffix_input = suffix_ids[:, :-1]

        out_suffix = model(input_ids=suffix_input, use_cache=False, past_key_values=copy.deepcopy(past_key_values))
        past_key_values = out_prefix.past_key_values

        prev_last_prefix_logits = last_prefix_logits
        last_prefix_logits = out_prefix.logits[:, -1, :]

        logits_for_loss = torch.cat([last_prefix_logits.unsqueeze(1),
                                out_suffix.logits],
                                dim=1)
        
        target = suffix_ids.reshape(-1).clone()
        target[:-(n_ground_truth_tokens + 1)] = -100
        target[-1] = -100

        nnll = torch.nn.functional.cross_entropy(
            input=logits_for_loss.reshape(-1, logits_for_loss.size(-1)),
            target=target,
            reduction="mean",
        ).item()
        # nnll = torch.nn.functional.cross_entropy(
        #     input=logits_for_loss.reshape(-1, logits_for_loss.size(-1)),
        #     target=target,
        #     reduction="none",
        # )#.item()

        # print(nnll.tolist())

        token_id = prefix_ids[:, -1:]
        token = tokenizer.decode(token_id[0])
        
        # Get logprob of the `t` token by shifting logprobs by -1
        if prev_last_prefix_logits is None:
            logits = out_prefix.logits[:, -2, :]
        else:
            logits = prev_last_prefix_logits
        all_token_logprobs = torch.nn.functional.log_softmax(logits, dim=-1)
        token_logprob = all_token_logprobs.gather(-1, token_id).squeeze(-1).item()


        tokens.append({
            'idx': t,
            'token_id': token_id[0].item(),
            'token': token,
            'nnll': nnll,
            'norm_coeff': n_ground_truth_tokens,
            'logprob': token_logprob
        })

    return pd.DataFrame(tokens)

In [19]:
expected_df = calc_nnll_scores(query, reasoning_trace, answer=expected_answer)
expected_df['answer_type'] ='Correct (Arthur Ignatius Conan Doyle)'

In [20]:
paraphrased_df = calc_nnll_scores(query, reasoning_trace, answer=paraphrased_answer)
paraphrased_df['answer_type'] = 'Re-raphrased (Arthur Conan Doyle)'

In [21]:
wrong_df = calc_nnll_scores(query, reasoning_trace, answer=wrong_answer)
wrong_df['answer_type'] = 'Incorrect alternative (Penelope Lively)'

In [22]:
# random_df = calc_nnll_scores(query, reasoning_trace, answer=random_answer)
# random_df['answer_type'] = 'random'

In [23]:
nnll_df = pd.concat([
    expected_df,
    paraphrased_df,
    wrong_df,
    # random_df
])

In [24]:
sample_row['query'], sample_row['ground_truth']

('Who was born first, Arthur Conan Doyle or Penelope Lively?',
 'Arthur Ignatius Conan Doyle')

In [25]:
_plot_df = nnll_df.copy()
_query = query
_ground_truth = expected_answer
_plot_title = f"NNLL and Smoothed NNLL by token relative position</b><br><b>Q</b>: {_query}<br><b>GT</b>: {_ground_truth}"

fig = px.line(
    _plot_df.sort_values(['answer_type', 'idx']),
    x='idx',
    y='nnll',
    color='answer_type',
    line_dash_map={'ground_truth': 'solid', 'wrong': 'dot'},
    title=_plot_title,
    labels={'idx': 'Token Position', 'nnll': 'NNLL'},
    height=500,
    width=1300,
)

# Обновляем hover с токеном в текстовом виде
fig.update_traces(
    hovertemplate=(
        "<b>Position:</b> %{x}<br>"
        "<b>NNLL:</b> %{y:.4f}<br>"
        "<b>Span:</b> %{customdata[0]}<extra></extra>"
    ),
    customdata=_plot_df[['token']].values
)

fig.update_layout(
    font=dict(
        family="Courier New, monospace",
        weight=550,
    ),
    showlegend=True
)

fig.update_xaxes(
    ticktext=_plot_df.sort_values(['answer_type', 'idx'])['token'],
    tickvals=_plot_df.sort_values(['answer_type', 'idx'])['idx'],
    tickangle=45,
    tickfont=dict(size=10)
)
fig.show()

In [ ]:
if not reasoning_trace.startswith(THINKING_START_TOKEN):
    _trace_for_display = f"{THINKING_START_TOKEN}\n{reasoning_trace}"
else:
    _trace_for_display = reasoning_trace
if _trace_for_display.endswith(THINKING_END_TOKEN):
    _trace_for_display = _trace_for_display[: -len(THINKING_END_TOKEN)].rstrip()

trace_tok = tokenizer(_trace_for_display, return_tensors="pt")
trace_token_ids = trace_tok.input_ids[0].tolist()
trace_tokens = [tokenizer.decode([tid]) for tid in trace_token_ids]

print(f"Reasoning trace length: {len(trace_token_ids)} tokens")
print("\n--- Reasoning trace ---")
print(_trace_for_display)

In [ ]:
truncation_token_idx = 200
# truncation_token_idx = 17

truncated_ids = trace_token_ids[:truncation_token_idx]
truncated_trace_str = tokenizer.decode(truncated_ids, skip_special_tokens=False)

print(f"Truncation at token index: {truncation_token_idx} / {len(trace_token_ids)}")
print(f"\nTruncated trace ({truncation_token_idx} tokens):")
print(truncated_trace_str)

prompts = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": query}]
context = tokenizer.apply_chat_template(
    prompts, tokenize=False, add_generation_prompt=True, enable_thinking=True
)


In [ ]:
answer_prefix = f"{TRUNCATED_TAG}\n{THINKING_END_TOKEN}\nAnswer: `"
# answer_prefix = f"\n{THINKING_END_TOKEN}\nAnswer: `"
model_input_str = context + truncated_trace_str + answer_prefix

print("--- Model input (last 500 chars) ---")
print(model_input_str[-500:])
print("\n" + "=" * 60)

input_ids = tokenizer(model_input_str, return_tensors="pt").input_ids.to(model.device)
print(f"Input length: {input_ids.shape[1]} tokens")

with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=64,
        do_sample=False,
        temperature=None,
        top_p=None,
    )

generated_ids = output_ids[0, input_ids.shape[1]:]
generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

print(f"\n--- Generated answer ---")
print(generated_text)